# MedNorm-VI ZS0 Zero-Shot Baseline and Submission

The canonical ZS0 notebook: a **pure zero-shot / pretrained** end-to-end baseline
that produces a valid organizer `output.zip`.

## What ZS0 is, and is not

It uses deterministic grammar, pinned pretrained checkpoints and locked
ontologies. It uses **no weights this project fitted** and **no head this project
initialized** — which excludes E3 (fine-tuned, exact F1 0.7103, Audit 0033), E4
(`RETIRED_FROM_ACTIVE_STACK`, Audit 0048) and E5 (randomly initialized MRC head,
which is not zero-shot but noise behind a confident interface).

No external API. No internet at runtime. No `internal_test`, ever.

## Three arms

    ZS0-A   E1 + E2                                   deterministic only
    ZS0-B   E1 + E2 + pretrained GLiNER
    ZS0-C   E1 + E2 + pretrained GLiNER + constrained Qwen on uncertain segments

## Stages

    0  install pinned dependencies; print environment and GPU
    1  verify configs, revisions, ontology snapshots, hashes, parameter ledger
    2  contract smoke tests on synthetic and golden examples
    3  run ZS0-A/B/C once on the governed validation split
    4  one comparison table
    5  select the submission arm by a documented deterministic rule
    6  organizer inference — DISABLED by default
    7  validate and package output.zip

Stages 0-5 are safe to re-run. Stage 6 writes the artifact that gets submitted,
so it is gated on the flag, the exact authorization string, and eight
preconditions checked in `assert_organizer_inference_allowed`.

## What is never done here

No training, no backward pass, no optimizer. No threshold search — every
resolver threshold is fixed in `configs/resolution/zs0_conservative_v1.yaml`. No
repeated inspection of organizer output: it is produced once, validated, hashed.


In [ ]:
# =============================================================================
# STAGE 0 - PINNED DEPENDENCIES. Runs before any project import.
# =============================================================================
%pip install -q gliner==0.2.13 transformers==4.44.2 sentence-transformers==3.0.1


In [ ]:
# =============================================================================
# STAGE 0b - ENVIRONMENT
# =============================================================================
import torch

from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = os.environ.get("MEDNORM_REPO_URL", "https://github.com/vquclinh/MedNorm-VI")
REPO_REF = os.environ.get("MEDNORM_REPO_REF", "main")
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
OUTPUT_DIR = Path("/content/zs0_output")
ORGANIZER_INPUT_DIR = DRIVE_ROOT / "organizer" / "round1_input"

# ---------------------------------------------------------------------------
# OPERATOR SETTINGS - committed fully disabled.
# ---------------------------------------------------------------------------
RUN_ORGANIZER_INFERENCE = False
CONFIRM_ORGANIZER_INFERENCE = ""

SEED = 20260728

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
except Exception as error:  # noqa: BLE001 - Colab-only import
    print(json.dumps({"stage": "drive_mount_skipped", "detail": str(error)}))

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True).stdout.strip()

environment = {
    "stage": "stage0_environment",
    "git_commit": GIT_COMMIT,
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
    "seed": SEED,
    "internal_test_accessed": False,
}
torch.manual_seed(SEED)
print(json.dumps(environment, indent=2, sort_keys=True))


In [ ]:
# =============================================================================
# STAGE 1 - CONFIGS, REVISIONS, ONTOLOGY SNAPSHOTS, HASHES, PARAMETER LEDGER
# =============================================================================
import yaml

from mednorm_vi.governance.e4_retirement import (
    E4RetirementRecord,
    assert_e4_absent_from_ledger,
    assert_e4_disabled,
    assert_no_e4_checkpoint_required,
)
from mednorm_vi.zs0 import (
    MAX_ACTIVE_PARAMETERS,
    LedgerEntry,
    OntologySnapshot,
    OrganizerPreconditions,
    all_arms,
    assert_organizer_inference_allowed,
    assert_zs0_components_allowed,
    assert_zs0_feature_flags,
    build_ledger,
    render_ledger,
    select_arm,
    validate_package,
)
from mednorm_vi.zs0.ledger import load_ledger
from mednorm_vi.zs0.profile import assert_no_external_api
from mednorm_vi.zs0.submission import package_hashes, validate_output_payload

CONFIG_PATH = REPO_DIR / "configs" / "pipeline" / "zs0_baseline.yaml"
LEDGER_PATH = REPO_DIR / "configs" / "models" / "zs0_parameter_ledger.yaml"
ZS0_CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

# E4 is retired: assert it, do not assume it.
assert_e4_disabled(ZS0_CONFIG["feature_flags"], profile="zs0_baseline")
assert_no_e4_checkpoint_required(
    ZS0_CONFIG.get("full_requires_checkpoints", []), profile="zs0_baseline")
# ZS0 uses no fitted weights and no random head.
assert_zs0_feature_flags(ZS0_CONFIG["feature_flags"])
for arm in all_arms():
    assert_zs0_components_allowed(arm.components)
assert_no_external_api(dict(os.environ))

def sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

HASHES = {
    "zs0_baseline.yaml": sha256_path(CONFIG_PATH),
    "zs0_parameter_ledger.yaml": sha256_path(LEDGER_PATH),
    "zs0_conservative_v1.yaml": sha256_path(
        REPO_DIR / "configs" / "resolution" / "zs0_conservative_v1.yaml"),
}
print(json.dumps({"stage": "stage1_config_hashes", **HASHES}, indent=2, sort_keys=True))
print(json.dumps({"stage": "stage1_e4_retirement",
                  **E4RetirementRecord().as_dict()}, indent=2, sort_keys=True))

# --- count every pretrained checkpoint programmatically -------------------
# The tracked ledger ships parameter_count: null so the gate fails CLOSED. The
# counts are produced HERE, from the checkpoints actually loaded.
def count_parameters(module):
    return int(sum(p.numel() for p in module.parameters()))

LEDGER_ENTRIES = list(load_ledger(LEDGER_PATH))
assert_e4_absent_from_ledger([e.component_id for e in LEDGER_ENTRIES])
COUNTED = {}
# Populate COUNTED[component_id] = (parameter_count, revision, sha256) after
# loading each model below, then rebuild the entries with verified counts.
print(json.dumps({"stage": "stage1_ledger_pending",
                  "components_awaiting_count": [
                      e.component_id for e in LEDGER_ENTRIES if not e.count_verified],
                  "max_active_parameters": MAX_ACTIVE_PARAMETERS,
                  "gate": "fails_closed_until_every_active_component_is_counted"},
                 indent=2, sort_keys=True))


In [ ]:
# =============================================================================
# STAGE 2 - CONTRACT SMOKE TESTS on synthetic and golden examples
#
# These run BEFORE any organizer document is touched. They exercise the offset
# invariant, the Qwen substring rules and the ontology constraint on inputs
# whose correct answers are known.
# =============================================================================
from mednorm_vi.zs0.linking import Candidate, select_from_candidates
from mednorm_vi.zs0.proposals import (
    ProposalRejected,
    convert_gliner_spans,
    qwen_proposals,
    resolve_occurrence,
)
from mednorm_vi.zs0.resolver import ResolverThresholds, resolve

smoke = {"stage": "stage2_contract_smoke", "checks": {}}

# 1. GLiNER offsets must reproduce the text, or the span is rejected.
text = "Bệnh nhân sốt cao và ho khan ."
good, ledger = convert_gliner_spans(
    document_id="smoke", original_text=text,
    raw_spans=[{"start": 10, "end": 17, "label": "symptom", "score": 0.9}],
    model_revision="pinned", checkpoint_sha256="0" * 64)
smoke["checks"]["gliner_valid_span_accepted"] = len(good) == 1
bad, ledger = convert_gliner_spans(
    document_id="smoke", original_text=text,
    raw_spans=[{"start": 0, "end": 5, "label": "symptom", "score": 0.9},
               {"start": 999, "end": 1002, "label": "symptom", "score": 0.9}],
    model_revision="pinned", checkpoint_sha256="0" * 64, ledger=ledger)
smoke["checks"]["gliner_out_of_range_rejected"] = ledger.counts.get(
    "span_outside_the_document", 0) >= 1

# 2. Qwen must return literal substrings; repeats fail closed without an anchor.
try:
    resolve_occurrence("sốt và sốt", "sốt")
    smoke["checks"]["ambiguous_repeat_fails_closed"] = False
except ProposalRejected as rejected:
    smoke["checks"]["ambiguous_repeat_fails_closed"] = (
        rejected.reason == "repeated_substring_without_a_disambiguating_anchor")
start, end = resolve_occurrence("sốt và sốt cao", "sốt", anchor="sốt cao")
smoke["checks"]["anchor_disambiguates"] = (start, end) == (7, 10)

invented, qledger = qwen_proposals(
    document_id="smoke", original_text=text, segment_start=0, segment_text=text,
    payload=json.dumps({"mentions": [{"text": "viêm phổi", "type": "DIAGNOSIS"}]}),
    model_revision="pinned", checkpoint_sha256="0" * 64)
smoke["checks"]["invented_mention_rejected"] = (
    len(invented) == 0
    and qledger.counts.get("not_a_literal_substring_of_the_segment", 0) == 1)

# 3. The resolver merges by coordinate identity, never by text.
resolved = resolve(good, thresholds=ResolverThresholds())
smoke["checks"]["resolver_preserves_provenance"] = all(
    m.provenance for m in resolved)

# 4. Qwen cannot introduce a code outside the offered candidate set.
snapshot = OntologySnapshot(
    ontology="ICD10", snapshot_id="smoke", codes=frozenset({"J18.9", "R50.9"}),
    aliases={"viêm phổi": ("J18.9",)})
offered = (Candidate(code="J18.9", ontology="ICD10", score=0.8,
                     source="lexical", rank=1),)
selected, rejected = select_from_candidates(
    json.dumps({"selected": ["A00.0"]}), offered, snapshot=snapshot)
smoke["checks"]["qwen_cannot_invent_a_code"] = (
    len(selected) == 0 and rejected == ("A00.0",))

smoke["all_passed"] = all(smoke["checks"].values())
print(json.dumps(smoke, indent=2, sort_keys=True))
if not smoke["all_passed"]:
    raise SystemExit("contract smoke tests failed; ZS0 must not proceed")


In [ ]:
# =============================================================================
# STAGE 3 - RUN ZS0-A, ZS0-B, ZS0-C ONCE on the governed validation split
#
# This stage is INDEPENDENT of organizer inference. It runs on a plain
# Run all with every organizer flag False.
#
# internal_test is never opened. Each arm runs exactly once: a comparison, not
# a search. An arm that predicts nothing still returns a real ArmResult with
# zero metrics; an arm that CANNOT run raises and names the prerequisite.
# =============================================================================
from mednorm_vi.deterministic_baseline.pipeline import run_phase1b
from mednorm_vi.document_intelligence import analyze_document, load_l1_config
from mednorm_vi.mention_factory.models import SpanProposal  # noqa: F401
from mednorm_vi.phase1c_foundation import Phase1BConfig
from mednorm_vi.training.phase2.e4.contracts import (
    E4_GOVERNED_VALIDATION_SHA256,
    resolve_governed_split_by_sha256,
)
from mednorm_vi.lattice.models import (
    EXPERT_LABORATORY_PARSER,
    EXPERT_MEDICATION_GRAMMAR,
    ExpertSpanProposal,
)
from mednorm_vi.zs0.backends import (
    GLiNERBackend,
    QwenBackend,
    build_mention_prompt,
    parse_completion,
)
from mednorm_vi.zs0.runner import (
    ArmSources,
    Document,
    assert_stage3_complete,
    run_all_arms,
)

VALIDATION = resolve_governed_split_by_sha256(
    split="validation", expected_sha256=E4_GOVERNED_VALIDATION_SHA256,
    search_roots=(DRIVE_ROOT / "data", REPO_DIR / "data"))
print(json.dumps({"stage": "stage3_corpus", "path": str(VALIDATION.path),
                  "sha256": VALIDATION.sha256, "internal_test_accessed": False},
                 indent=2, sort_keys=True))

# --- governed documents ---------------------------------------------------
DOCUMENTS = []
with VALIDATION.path.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        row = json.loads(line)
        DOCUMENTS.append(Document(
            document_id=str(row["document_id"]),
            text=str(row["text"]),
            gold=tuple(
                (int(e["start"]), int(e["end"]),
                 str(e.get("target_type") or e.get("type")))
                for e in row.get("entities", []))))
print(json.dumps({"stage": "stage3_documents", "count": len(DOCUMENTS),
                  "gold_mentions": sum(len(d.gold) for d in DOCUMENTS)},
                 indent=2, sort_keys=True))

# --- E1 + E2, the real deterministic path ---------------------------------
L1_CONFIG, L1_LEXICON = load_l1_config(str(REPO_DIR / ZS0_CONFIG["l1_config"]))
PHASE1B_CONFIG = Phase1BConfig.load(
    str(REPO_DIR / ZS0_CONFIG["router_config"]),
    str(REPO_DIR / ZS0_CONFIG["medication_config"]),
    str(REPO_DIR / ZS0_CONFIG["laboratory_config"]))

def deterministic_proposals(document):
    """E1 medication grammar + E2 laboratory parser over one document."""
    scratch = Path("/content/zs0_scratch")
    scratch.mkdir(parents=True, exist_ok=True)
    source = scratch / f"{document.document_id.replace(':', '_')}.txt"
    source.write_text(document.text, encoding="utf-8")
    graph = analyze_document(source, config=L1_CONFIG, lexicon=L1_LEXICON)
    result = run_phase1b(graph, PHASE1B_CONFIG)
    converted = []
    for ordinal, proposal in enumerate(result.proposals, start=1):
        start, end = int(proposal.start), int(proposal.end)
        if document.text[start:end] != proposal.text:
            continue   # the offset invariant is absolute
        converted.append(ExpertSpanProposal(
            document_id=document.document_id, start=start, end=end,
            text=proposal.text,
            type_scores={proposal.entity_type: float(proposal.score)},
            local_score=float(proposal.score),
            expert_id=(EXPERT_LABORATORY_PARSER
                       if proposal.entity_type in ("TEST_NAME", "TEST_RESULT")
                       else EXPERT_MEDICATION_GRAMMAR),
            proposal_id=f"zs0-det-{document.document_id}-{ordinal:04d}",
            original_start=start, original_end=end))
    return converted

# --- pretrained backends, loaded and COUNTED ------------------------------
GLINER = GLiNERBackend(cache_dir=str(MODEL_CACHE_DIR)).load()
COUNTED["e6_gliner"] = GLINER.parameter_count()
QWEN = QwenBackend(cache_dir=str(MODEL_CACHE_DIR)).load()
COUNTED["e7_qwen_cascade"] = QWEN.parameter_count()
print(json.dumps({"stage": "stage3_backends_loaded",
                  "parameter_counts": COUNTED}, indent=2, sort_keys=True))

def qwen_mentions(segment: str) -> str:
    return parse_completion(QWEN.generate(build_mention_prompt(segment)))

# --- rebuild the ledger with the counts we just measured ------------------
VERIFIED_ENTRIES = []
for entry in LEDGER_ENTRIES:
    count = COUNTED.get(entry.component_id)
    if count is None:
        VERIFIED_ENTRIES.append(entry)
        continue
    VERIFIED_ENTRIES.append(LedgerEntry(
        component_id=entry.component_id, model_id=entry.model_id,
        revision=entry.revision, checkpoint_sha256=entry.checkpoint_sha256,
        parameter_count=count, adapter_parameters=entry.adapter_parameters,
        loaded_during_inference=entry.loaded_during_inference,
        concurrent_with_other_models=entry.concurrent_with_other_models,
        shared_instance_key=entry.shared_instance_key,
        count_method="counted_from_checkpoint", count_verified=True))
LEDGER_ENTRIES = VERIFIED_ENTRIES
ACTIVE_LEDGER = build_ledger(
    [e for e in LEDGER_ENTRIES if e.count_verified or not e.loaded_during_inference],
    enforce=True)
print(render_ledger(ACTIVE_LEDGER))

# --- run all three arms ---------------------------------------------------
ARM_SOURCES = {
    "ZS0-A": ArmSources(deterministic=deterministic_proposals),
    "ZS0-B": ArmSources(deterministic=deterministic_proposals,
                        gliner=GLINER.predict,
                        gliner_revision=GLINER.revision or GLINER.model_id,
                        gliner_sha256=""),
    "ZS0-C": ArmSources(deterministic=deterministic_proposals,
                        gliner=GLINER.predict,
                        gliner_revision=GLINER.revision or GLINER.model_id,
                        gliner_sha256="",
                        qwen=qwen_mentions,
                        qwen_revision=QWEN.revision or QWEN.model_id,
                        qwen_sha256=""),
}
ACTIVE_BY_ARM = {
    "ZS0-A": 0,
    "ZS0-B": COUNTED.get("e6_gliner", 0),
    "ZS0-C": COUNTED.get("e6_gliner", 0) + COUNTED.get("e7_qwen_cascade", 0),
}

# No try/except here on purpose: a broken arm must stop the run and name
# itself, not leave a short ARM_RESULTS for Stage 4 to trip over.
ARM_RESULTS = run_all_arms(
    list(all_arms()), DOCUMENTS, ARM_SOURCES,
    active_parameters_by_arm=ACTIVE_BY_ARM)
assert_stage3_complete(ARM_RESULTS, list(all_arms()))
print(json.dumps({"stage": "stage3_complete",
                  "arm_results": len(ARM_RESULTS),
                  "arms": sorted(r.arm for r in ARM_RESULTS)},
                 indent=2, sort_keys=True))


In [ ]:
# =============================================================================
# STAGE 4 - ONE COMPARISON TABLE
# =============================================================================
if not ARM_RESULTS:
    raise SystemExit("Stage 3 produced no arm results")

header = ("| arm | exact P | exact R | exact F1 | wrong-type | malformed | "
          "offset viol | runtime s | VRAM GiB | active params |")
rule = "| " + " | ".join(["---"] * 10) + " |"
rows = [
    f"| {r.arm} | {r.exact_precision:.4f} | {r.exact_recall:.4f} | "
    f"{r.exact_f1:.4f} | {r.wrong_type_count} | {r.malformed_proposal_count} | "
    f"{r.offset_violations} | {r.runtime_seconds:.1f} | {r.peak_vram_gib:.2f} | "
    f"{r.active_parameters:,} |"
    for r in ARM_RESULTS
]
print("\n".join([header, rule, *rows]))
print(json.dumps({"stage": "stage4_comparison",
                  "results": [r.as_dict() for r in ARM_RESULTS]},
                 indent=2, sort_keys=True))


In [ ]:
# =============================================================================
# STAGE 5 - DETERMINISTIC ARM SELECTION
# =============================================================================
SELECTED_ARM, SELECTION_REPORT = select_arm(ARM_RESULTS)
print(json.dumps({"stage": "stage5_selection", **SELECTION_REPORT},
                 indent=2, sort_keys=True))
if SELECTED_ARM is None:
    raise SystemExit(SELECTION_REPORT["reason"])


In [ ]:
# =============================================================================
# STAGE 6 - ORGANIZER INFERENCE.  DISABLED BY DEFAULT.
#
# Every precondition is checked before a single document is read.
# =============================================================================
ORGANIZER_FILES = sorted(ORGANIZER_INPUT_DIR.glob("*.txt")) if (
    ORGANIZER_INPUT_DIR.is_dir()) else []

LEDGER_REPORT = build_ledger(LEDGER_ENTRIES, enforce=False)
PRECONDITIONS = OrganizerPreconditions(
    selected_arm=SELECTED_ARM.arm if SELECTED_ARM else None,
    active_trained_experts=tuple(
        flag for flag in ("enable_e3_vihealthbert", "enable_e4_phobert_w2ner",
                          "enable_e5_xlmr_mrc")
        if ZS0_CONFIG["feature_flags"].get(flag)),
    training_executed=False,
    ledger=LEDGER_REPORT,
    local_gates_passed=bool(smoke["all_passed"]),
    discovered_documents=tuple(p.name for p in ORGANIZER_FILES),
    external_api_configured=False,
    seed=SEED,
    manifest_recorded=True)
print(json.dumps({"stage": "stage6_preconditions", **PRECONDITIONS.as_dict()},
                 indent=2, sort_keys=True))
print(render_ledger(LEDGER_REPORT))

assert_organizer_inference_allowed(
    enabled=RUN_ORGANIZER_INFERENCE,
    confirmation=CONFIRM_ORGANIZER_INFERENCE,
    preconditions=PRECONDITIONS)

# Run the SELECTED arm over the 100 organizer documents and write
# OUTPUT_DIR/1.json .. OUTPUT_DIR/100.json.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({"stage": "stage6_authorized", "arm": SELECTED_ARM.arm,
                  "documents": len(ORGANIZER_FILES),
                  "output_dir": str(OUTPUT_DIR)}, indent=2, sort_keys=True))


In [ ]:
# =============================================================================
# STAGE 7 - VALIDATE AND PACKAGE
# =============================================================================
import zipfile

problems = []
for path in sorted(ORGANIZER_FILES):
    name = f"{int(path.stem)}.json"
    payload_path = OUTPUT_DIR / name
    problems.extend(validate_output_payload(
        name, payload_path.read_text(encoding="utf-8"),
        original_text=path.read_text(encoding="utf-8")))
if problems:
    print(json.dumps({"stage": "stage7_validation_failed",
                      "problems": problems[:20]}, indent=2, sort_keys=True))
    raise SystemExit(f"{len(problems)} output validation problems; not packaging")

ZIP_PATH = OUTPUT_DIR.parent / "output.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for index in range(1, 101):
        archive.write(OUTPUT_DIR / f"{index}.json", arcname=f"{index}.json")

package_problems = validate_package(ZIP_PATH)
if package_problems:
    raise SystemExit(f"package validation failed: {package_problems}")

HASH_REPORT = package_hashes(OUTPUT_DIR, ZIP_PATH)
MANIFEST = {
    "stage": "stage7_package",
    "selected_arm": SELECTED_ARM.arm,
    "git_commit": GIT_COMMIT,
    "seed": SEED,
    "config_hashes": HASHES,
    "ledger": LEDGER_REPORT.as_dict(),
    "output_zip": str(ZIP_PATH),
    "output_zip_sha256": HASH_REPORT["output.zip"],
    "output_json_sha256": {k: v for k, v in HASH_REPORT.items() if k != "output.zip"},
    "internal_test_accessed": False,
}
(OUTPUT_DIR.parent / "zs0_run_manifest.json").write_text(
    json.dumps(MANIFEST, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({k: v for k, v in MANIFEST.items()
                  if k != "output_json_sha256"}, indent=2, sort_keys=True))
print("output.zip SHA-256:", HASH_REPORT["output.zip"])


## Return-to-repository

Record in the next append-only audit:

* the Stage-4 comparison table for ZS0-A/B/C;
* the Stage-5 selected arm and the deterministic reason;
* the Stage-1 parameter ledger with every verified count and the active total;
* `output.zip` path and SHA-256 from `zs0_run_manifest.json`.

Report the measured numbers exactly. **Do not claim a leaderboard score** — that
belongs to the organizer after the human uploads `output.zip`.
